In [ ]:
import os
import pandas as pd
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
from torch import nn
from torchvision import models
from tqdm import tqdm

# Chemins des fichiers et répertoires
model_path = r"D:\wealth_predict_sentinel\models\model_vgg16_fullyConnected_sans_augm_couche_nongelee_batch_16.pt"
test_image_dir = r"D:\wealth_predict_sentinel\Data\downloaded\Image_satellite_base_EHCVM_2018_Zoom_14_Sentinel_2_pour_an_2021" # menage EHCVM 2018 leurs images en 2021
csv_path = r"D:\wealth_predict_sentinel\Data\processed_csv\Data_EHCVM_2018_with_images_names.csv"
output_path = r"D:\wealth_predict_sentinel\Data\processed_csv\images_with_features_extracted_4096_fullyConnected_sans_augm_couche_nongelee_batch_16_base_EHCVM_2018_pour_an_2021.csv" #On met à jour ce nom quand on met de nouvelles données

# Définir l'architecture exactement comme dans l'entraînement
class VGG16Standard(nn.Module):
    def __init__(self, num_classes=4):
        super(VGG16Standard, self).__init__()
        self.vgg16 = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        
        # Modifier uniquement la dernière couche pour notre nombre de classes
        num_features = self.vgg16.classifier[-1].in_features
        self.vgg16.classifier[-1] = nn.Linear(num_features, num_classes)

    def get_features(self, x):
        # Passer à travers les couches convolutives
        x = self.vgg16.features(x)
        x = self.vgg16.avgpool(x)
        x = torch.flatten(x, 1)
        
        # Passer à travers toutes les couches du classifier sauf la dernière
        for layer in list(self.vgg16.classifier.children())[:-1]:
            x = layer(x)
        return x

# Transformation des images (identique à l'entraînement)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Fonction pour charger une image
def load_image(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        return transform(image)
    except Exception as e:
        print(f"Erreur lors du chargement de l'image {image_path}: {e}")
        return None

print("Configuration du device...")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device: {device}")

print("Chargement du modèle...")
model = VGG16Standard(num_classes=4).to(device)
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Vérification des dimensions
print("Vérification des dimensions des caractéristiques...")
dummy_image = torch.randn(1, 3, 224, 224).to(device)
dummy_features = model.get_features(dummy_image)
print(f"Dimensions des caractéristiques : {dummy_features.shape}")  # Devrait être [1, 4096]

print("Chargement des données...")
df = pd.read_csv(csv_path)
total_images = len(df)
print(f"Nombre total d'images à traiter : {total_images}")

# Extraction des caractéristiques
print("Début de l'extraction des caractéristiques...")
features_list = []

for idx, row in tqdm(df.iterrows(), total=total_images, desc="Extraction des caractéristiques"):
    image_name = row['nom de l\'image']
    image_path = os.path.join(test_image_dir, image_name)
    
    if os.path.exists(image_path):
        image_tensor = load_image(image_path)
        if image_tensor is not None:
            # Extraction des caractéristiques
            with torch.no_grad():
                image_tensor = image_tensor.unsqueeze(0).to(device)
                features = model.get_features(image_tensor)
                features = features.cpu().numpy().flatten()
        else:
            features = np.zeros(4096)
            print(f"Image invalide : {image_name}")
    else:
        features = np.zeros(4096)
        print(f"Image introuvable : {image_name}")
    
    features_list.append(features)

# Création du DataFrame avec les caractéristiques
print("Création du DataFrame avec les caractéristiques...")
features_df = pd.DataFrame(features_list, columns=[f"feature_{i}" for i in range(4096)])

# Combinaison avec le DataFrame original
print("Combinaison avec les données originales...")
df_with_features = pd.concat([df.reset_index(drop=True), features_df.reset_index(drop=True)], axis=1)

# Sauvegarde des résultats
print(f"Sauvegarde du DataFrame sous : {output_path}")
df_with_features.to_csv(output_path, index=False)

print("\nStatistiques finales:")
print(f"Nombre total d'images traitées : {total_images}")
print(f"Dimensions du DataFrame final : {df_with_features.shape}")
print("Terminé!")

In [ ]:
pd.read_csv(r"D:\wealth_predict_sentinel\Data\processed_csv\images_with_features_extracted_4096_fullyConnected_sans_augm_couche_nongelee_batch_16_base_EHCVM_2018_pour_an_2021.csv")
